In [33]:
# Cell 1: Imports

import hashlib
from random import randint

In [34]:
# Cell 2: Parameter Generation

def generate_params(bits=16):
    """
    Generates parameters for Schnorr:
    A prime p and a generator g of the group.
    """
    # 1. Generate a 'Safe Prime' p where p = 2q + 1 and q is also prime.
    # This prevents the Pohlig-Hellman algorithm from attacking the DLP.
    
    # We use Sage's 'is_prime' to ensure the prime is strong.
    found = False
    while not found:
        # Generate a random prime q
        q = random_prime(2^bits, lbound=2^(bits-1))
        p = 2*q + 1
        if is_prime(p):
            found = True
            
    # 2. Find a generator g
    # In a safe prime group, we often want a generator of the 
    # subgroup of quadratic residues to ensure maximum security.
    # Sage's primitive_root works perfectly here.
    g = primitive_root(p)
    
    return p, g

In [35]:
# Cell 3: Key Generation

def keygen(p, g):
    x = randint(1, p-2)          # private key
    y = power_mod(g, x, p)       # public key
    return x, y

In [36]:
# Cell 4: Hash Function

def hash_function(r, m, p):
    # Encoding to bytes ensures consistency across different systems
    data = f"{r}|{m}".encode()
    h = hashlib.sha256(data).hexdigest()
    # We take the hash modulo (p-1) to ensure the exponent stays in range
    return int(h, 16) % (p - 1)

In [37]:
# Cell 5: Signing

def sign(p, g, x, m):
    # k must be coprime to p-1 for maximum security, but random is okay for demo
    k = randint(1, p - 2)
    
    # Commitment
    r = power_mod(g, k, p)
    
    # Challenge
    e = hash_function(r, m, p)
    
    # Response: s = (k + x*e) mod (p-1)
    s = (k + x * e) % (p - 1)
    
    return r, s

In [38]:
# Cell 6: Verification

def verify(p, g, y, m, r, s):
    e = hash_function(r, m, p)
    
    # Verification equation: g^s == r * y^e (mod p)
    left_side = power_mod(g, s, p)
    right_side = (r * power_mod(y, e, p)) % p
    
    return left_side == right_side

In [40]:
# Cell 7: Main Execution (Interactive)

print("🔐 Schnorr Signature Scheme Demo\n")

# Step 1: Parameters
p, g = generate_params()

# Step 2: Keys
x, y = keygen(p, g)

print("Public Key (y):", y)
print("--------------------------------------------------")

# Step 3: User Input
m = input("✍️ Enter your message: ")

# Step 4: Signing
r, s = sign(p, g, x, m)

print("\n🔏 Signature Generated:")
print("r =", r)
print("s =", s)

print("--------------------------------------------------")

# Step 5: Verification
valid = verify(p, g, y, m, r, s)

if valid:
    print("✅ Signature is VALID")
else:
    print("❌ Signature is INVALID")

🔐 Schnorr Signature Scheme Demo

Public Key (y): 89902
--------------------------------------------------


✍️ Enter your message:  hi



🔏 Signature Generated:
r = 81285
s = 50822
--------------------------------------------------
✅ Signature is VALID


In [41]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create the UI elements
msg_input = widgets.Text(value='Hello Schnorr', description='Message:')
bit_slider = widgets.IntSlider(value=16, min=8, max=32, step=1, description='Prime Bits:')
run_button = widgets.Button(description='Generate & Sign', button_style='success')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        # 1. Params
        p_val, g_val = generate_params(bits=bit_slider.value)
        # 2. Keys
        x_val, y_val = keygen(p_val, g_val)
        # 3. Sign
        r_val, s_val = sign(p_val, g_val, x_val, msg_input.value)
        # 4. Verify
        is_valid = verify(p_val, g_val, y_val, msg_input.value, r_val, s_val)
        
        print(f"Prime (p): {p_val}")
        print(f"Public Key (y): {y_val}")
        print(f"Signature (r, s): ({r_val}, {s_val})")
        print(f"Result: {'✅ Valid' if is_valid else '❌ Invalid'}")

run_button.on_click(on_button_clicked)
display(bit_slider, msg_input, run_button, output)

IntSlider(value=16, description='Prime Bits:', max=32, min=8)

Text(value='Hello Schnorr', description='Message:')

Button(button_style='success', description='Generate & Sign', style=ButtonStyle())

Output()